# Exploración breve con ydata-profiling
Perfilamos un subconjunto de 11 columnas de las 68 filas: identidad, procedencia, estado, potencia, torques y corrientes.
Generamos **dos perfiles separados**: referencia manual y salida del OCR seleccionado. Así una distribución razonable no disfraza errores del motor.

Los valores vacíos pueden ser estructurales, N/A explícitos u omisiones OCR. No imputar ceros ni concluir que todo vacío es un fallo.
No hacemos correlaciones ni una exploración extensa para un catálogo pequeño.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/ocr_lab.py").is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Markdown, Image
print("Python:", sys.executable)
print("Proyecto:", ROOT)

Python: C:\Users\nrir\Desktop\dev\serie76-assistant\.venv-ocr\Scripts\python.exe
Proyecto: C:\Users\nrir\Desktop\dev\serie76-assistant


In [2]:
from scripts.ocr_profile import profile
from datetime import datetime, timezone
selection = json.loads((ROOT / "config/ocr.json").read_text(encoding="utf-8"))
run_dir = ROOT / selection["benchmark"]
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
out = ROOT / "reports/ocr" / f"profile-{stamp}"
reference, extracted, summary = profile(run_dir, out)
display(pd.DataFrame([summary]))
display(reference.groupby(["source_table","data_status"], dropna=False).size().rename("filas"))
display(pd.DataFrame({"vacios_referencia": reference.isna().sum(),
                      "vacios_ocr": extracted.isna().sum()}))
print("Informes HTML:", out)

C:\Users\nrir\Desktop\dev\serie76-assistant\.venv-ocr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0/11 [00:00<?, ?it/s]

100%|██████████| 11/11 [00:00<00:00, 124.15it/s]

  0%|          | 0/11 [00:00<?, ?it/s]

 91%|█████████ | 10/11 [00:00<00:00, 94.70it/s]

100%|██████████| 11/11 [00:00<00:00, 95.33it/s]

,reference_rows,ocr_rows,reference_unique_codes,ocr_unique_codes,reference_explicit_NA,ocr_explicit_NA,engine,invalid_numeric_tokens,note
0,68,68,68,68,4,4,rapidocr,"{'motor_power_w': 0, 'on_off_torque_nm': 1, 'm...","Nulls can be structural, explicit N/A or OCR o..."


source_table  data_status
p1_t1         reported       11
p1_t2         reported       16
p2_t1         reported        7
              source_NA       4
p3_t1         reported        5
p3_t2         reported        5
p4_t1         reported       10
p4_t2         reported       10
Name: filas, dtype: int64

,vacios_referencia,vacios_ocr
source_table,0,0
source_row,0,0
base_part_number,0,0
data_status,0,64
motor_power_w,4,4
on_off_torque_nm,16,17
modulating_torque_nm,16,16
shared_torque_nm,61,61
fla_60hz_a,14,14
fla_50hz_a,14,14


Informes HTML: C:\Users\nrir\Desktop\dev\serie76-assistant\reports\ocr\profile-20261103T020339284158Z


## Hallazgos y límites
El catálogo combina esquemas distintos: torque compartido en trifásico, servicios separados en otras tablas y corriente sin frecuencia en 24 V. Los cuatro N/A explícitos no son cero. El perfil describe esos patrones; la comprobación por celda del notebook 01 decide si el OCR conserva los valores.

In [3]:
from IPython.display import FileLink
display(FileLink(str((out / "reference_profile.html").relative_to(ROOT)), url_prefix="../"))
display(FileLink(str((out / "ocr_profile.html").relative_to(ROOT)), url_prefix="../"))
from scripts.ocr_lab import write_json
write_json(ROOT / "reports/ocr/latest_profile.json",
           {"directory": str(out.relative_to(ROOT)), **summary})
display(extracted.head(5))

C:\Users\nrir\Desktop\dev\serie76-assistant\reports\ocr\profile-20261103T020339284158Z\reference_profile.html

C:\Users\nrir\Desktop\dev\serie76-assistant\reports\ocr\profile-20261103T020339284158Z\ocr_profile.html

field,source_table,source_row,base_part_number,data_status,motor_power_w,on_off_torque_nm,modulating_torque_nm,shared_torque_nm,fla_60hz_a,fla_50hz_a,fla_unqualified_a
0,p1_t1,1,761A00-11300000/A,NaN,15.0,80.0,65.0,NaN,1.2,1.1,NaN
1,p1_t1,2,761B00-11300000/A,NaN,15.0,100.0,70.0,NaN,1.3,1.2,NaN
2,p1_t1,3,762A00-11300000/A,NaN,40.0,150.0,130.0,NaN,2.3,2.3,NaN
3,p1_t1,4,762B00-11300000/A,NaN,40.0,200.0,150.0,NaN,2.3,2.3,NaN
4,p1_t1,5,763A00-11300000/A,NaN,40.0,300.0,220.0,NaN,2.6,2.5,NaN
